In [5]:
import re

class FOLProver:
    def __init__(self):
        self.facts = set()
        self.rules = []
        self.proof_history = {}

    def parse_literal(self, s):
        s = s.strip()
        match = re.match(r'^([a-zA-Z_][a-zA-Z0-9_]*)\s*\((.*)\)$', s)
        if not match:
            raise ValueError(f"Invalid literal format: {s}")
        pred = match.group(1)
        args = tuple(arg.strip() for arg in match.group(2).split(',') if arg.strip())
        return (pred, args)

    def add_fact(self, fact_str):
        pred, args = self.parse_literal(fact_str)
        self.facts.add((pred, args))
        if (pred, args) not in self.proof_history:
            self.proof_history[(pred, args)] = ("Given Fact", [])

    def add_rule(self, rule_str):
        if '->' not in rule_str:
            raise ValueError("Rule must contain '->'")
        lhs_str, rhs_str = rule_str.split('->')

        lhs_literals = [self.parse_literal(lit) for lit in lhs_str.split('&')]
        rhs_literal = self.parse_literal(rhs_str)

        self.rules.append((lhs_literals, rhs_literal))

    def unify(self, pattern, datum, theta=None):
        if theta is None:
            theta = {}

        if pattern[0] != datum[0] or len(pattern[1]) != len(datum[1]):
            return None

        new_theta = theta.copy()
        for p_arg, d_arg in zip(pattern[1], datum[1]):
            is_var = p_arg.islower() and len(p_arg) == 1
            if is_var:
                if p_arg in new_theta:
                    if new_theta[p_arg] != d_arg:
                        return None
                else:
                    new_theta[p_arg] = d_arg
            else:
                if p_arg != d_arg:
                    return None
        return new_theta

    def match_rule_lhs(self, lhs_literals, index=0, theta=None):
        if theta is None:
            theta = {}

        if index == len(lhs_literals):
            yield theta
            return

        curr_pattern = lhs_literals[index]

        substituted_args = tuple(theta.get(arg, arg) for arg in curr_pattern[1])
        temp_pattern = (curr_pattern[0], substituted_args)

        for fact in list(self.facts):
            local_theta = self.unify(temp_pattern, fact, theta)
            if local_theta is not None:
                yield from self.match_rule_lhs(lhs_literals, index + 1, local_theta)

    def forward_reasoning(self, query_str):
        query = self.parse_literal(query_str)

        if query in self.facts:
            return True

        new_derived = True
        while new_derived:
            new_derived = False

            for lhs, rhs in self.rules:
                for theta in self.match_rule_lhs(lhs):
                    derived_args = tuple(theta.get(arg, arg) for arg in rhs[1])
                    derived_fact = (rhs[0], derived_args)

                    if derived_fact not in self.facts:
                        self.facts.add(derived_fact)
                        premises = []
                        for cond in lhs:
                            premise_args = tuple(theta.get(arg, arg) for arg in cond[1])
                            premises.append((cond[0], premise_args))

                        rule_desc = f"{' & '.join([f'{l[0]}({",".join(l[1])})' for l in lhs])} -> {rhs[0]}({','.join(rhs[1])}) with {theta}"
                        self.proof_history[derived_fact] = (rule_desc, premises)
                        new_derived = True

                        if derived_fact == query:
                            return True
        return query in self.facts

    def print_proof_tree(self, fact, depth=0):
        indent = "  " * depth
        fact_str = f"{fact[0]}({','.join(fact[1])})"
        if fact not in self.proof_history:
            print(f"{indent}- {fact_str} (Unknown / Not proved)")
            return

        rule_desc, premises = self.proof_history[fact]
        if rule_desc != "Given Fact":
            for premise in premises:
                self.print_proof_tree(premise, depth + 1)
            print(f"{indent}└─ {fact_str} [Derived via: {rule_desc}]")
        else:
            print(f"{indent}└─ {fact_str} (Given Fact)")



prover = FOLProver()

prover.add_fact("American(Robert)")
prover.add_fact("Enemy(CountryA, America)")
prover.add_fact("Missile(M1)")
prover.add_fact("Owns(CountryA, M1)")

prover.add_rule("Missile(x) -> Weapon(x)")
prover.add_rule("Enemy(x, America) -> Hostile(x)")
prover.add_rule("Owns(CountryA, x) & Missile(x) -> Sells(Robert, x, CountryA)")
prover.add_rule("American(x) & Weapon(y) & Sells(x, y, z) & Hostile(z) -> Criminal(x)")

query_to_prove = "Criminal(Robert)"

print(f"--- Attempting to prove: {query_to_prove} ---")
success = prover.forward_reasoning(query_to_prove)

if success:
    print("\nProof successfully established!\n")
    print("--- PROOF TREE ---")
    query_fact = prover.parse_literal(query_to_prove)
    prover.print_proof_tree(query_fact)
else:
    print("\nCould not prove the query with current facts and rules.")

--- Attempting to prove: Criminal(Robert) ---

Proof successfully established!

--- PROOF TREE ---
  └─ American(Robert) (Given Fact)
    └─ Missile(M1) (Given Fact)
  └─ Weapon(M1) [Derived via: Missile(x) -> Weapon(x) with {'x': 'M1'}]
    └─ Owns(CountryA,M1) (Given Fact)
    └─ Missile(M1) (Given Fact)
  └─ Sells(Robert,M1,CountryA) [Derived via: Owns(CountryA,x) & Missile(x) -> Sells(Robert,x,CountryA) with {'x': 'M1'}]
    └─ Enemy(CountryA,America) (Given Fact)
  └─ Hostile(CountryA) [Derived via: Enemy(x,America) -> Hostile(x) with {'x': 'CountryA'}]
└─ Criminal(Robert) [Derived via: American(x) & Weapon(y) & Sells(x,y,z) & Hostile(z) -> Criminal(x) with {'x': 'Robert', 'y': 'M1', 'z': 'CountryA'}]
